# Implementação do Modelo - Fase 2

Reogarnização dos arquivos do projeto para que reflitam as fases aplicadas na pesquisa. Conforme Guia_Fase2_Passo2_Estrutura_Modulos.cd

In [2]:
%load_ext autoreload
%autoreload 2


import random

from philder import infra
from philder import database_consolidation as consolidacao
from philder import data_processing as processamento
from philder import database_balance as balanceamento
from philder import performance_assessment as avaliacao


# --- Preparação ---
infra.criar_estrutura_diretorios()

# --- Módulo A: consolidação da base ---
#brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
#brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")
#df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

# --- Módulo B: processamento de texto ---
# df = processamento.pipeline_processamento(df)

# --- Módulo C: balanceamento + treino + teste ---
# df_balanceado = balanceamento.aplicar_undersampling(df)
# df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)
# tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
# dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
# trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
# predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
# balanceamento.salvar_modelo(trainer, tokenizer)

# --- Módulo D: avaliação de desempenho ---
# metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
# eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
# avaliacao.gerar_curva_perda(trainer.state.log_history)
# avaliacao.salvar_resultados({**metricas, **eficiencia})


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Estrutura de diretórios criada/confirmada:
  - dados_philder/
  - modelo/  (com modelo\checkpoints/)
  - resultados/


# A) DATABASE CONSOLIDATION

<DESENVOLVER>

In [3]:
# --- Módulo A: consolidação da base ---
brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")

# 19/09/2026 -  Ajuste da base para refletir a quantidade de amostras do artigo original, especifcamente
# da base do phishtank
# Nazario tem mais e-mails de phishing do que o PhishTank usado no artigo (501) —
# reduz por amostragem aleatória (sem reposição) para equiparar a mesma quantidade.
# SpamAssassin já está na mesma contagem do artigo (2.501), não precisa de ajuste.
random.seed(42)
brutos_phishing = random.sample(brutos_phishing, 501)

df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

Total de e-mails legítimos carregados: 2501
Total de e-mails de phishing carregados: 1565 (coluna: 'body')
Dataset consolidado: 3002 registros — legítimos: 2501, phishing: 501


# B) DATA PROCESSING
<DISCORRER MAIS>

In [4]:
# --- Módulo B: processamento de texto ---
df = processamento.pipeline_processamento(df)

Pré-processamento concluído: 3002 registros (16 ficaram vazios)


# C) DATABASE BALANCE

In [5]:
# --- Módulo C: balanceamento + treino + teste ---
df_balanceado = balanceamento.aplicar_undersampling(df)
df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)
tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
balanceamento.salvar_modelo(trainer, tokenizer)

Balanceamento aplicado: {1: 501, 0: 501}
Treino: {1: 351, 0: 350}
Teste: {0: 151, 1: 150}


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

Map:   0%|          | 0/701 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

\\?\C:\Users\alvar\AppData\Roaming\jupyterlab-desktop\envs\env_philder1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (3 outputs left out)

\\?\C:\Users\alvar\AppData\Roaming\jupyterlab-desktop\envs\env_philder1\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Tempo total de inferência no teste: 4.53s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo em: modelo/


# D) PERFORMANCE ASSESSMENT
## Alterações identificadas:

- [x] Incluir na tabela de métricas o Tempo de treino e o tempo de teste e tempo de detecção
- [ ] Identificar padrão de unidades usada no artigo para avaliar ajuste e necessidade de conversão de dados, pois os resultados são bem diferentes. Considerar também o hardware utilizado
- [ ]  Finalizado ajustes rodar notebook no google colabs e comparar os resultados IES, TES, RTDE
- [x] Executar um treinamento com o dataset atual, simulando a quantidade de amostras do artigo original

## Resultados do artigo original:

### Tabela 1 — Métricas de Desempenho Preditivo (PHILDER = TinyBERT (2), execução com undersampling)

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| TinyBERT (2) | 0,9927 | 0,9896 | 0,9960 | 0,9927 | 111,83 | 4,48 | 14,8841 |

### Tabela 2 — Métricas de Eficiência Computacional (PHILDER = TinyBERT + undersampling, valores brutos dos gráficos)
**Valores não normalizados**
| Métrica | Valor bruto |
|---|---|
| TES | 0,0389 |
| IES | 1,0788 |
| RTDE | 324,7750 |

## Resultados deste Trabalho Fase2

### Tabela 1 — Métricas de Desempenho Preditivo

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Modelo | Acc. | Prec. | Rec. | F1 |
|---|---|---|---|---|
| philder |0,9936 | 0,9957 | 0,9915 | 0,9936 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**
| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9867 | 1,0 | 0,9733 | 0,9865 | 255.8025 | 4.5329 | 15.0595 |


### Tabela 2 — Métricas de Eficiência Computacional

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,00268 |
| IES | 0,1474 |
| RTDE | 138,3831 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0039 |
| IES | 0,2177 |
| RTDE | 65,505 |

## Análise Crítica da Avaliação dos Resultados

- Até o momento os resultados de Eficiência são da ordem de 10x piores que o do artigo, indicando que as etapas de treino e testes foram executadoas em um ambiente de poder computacional elevado. O fator mais relevante para a discrepância nos valores foi o tempo. Avaliar este ponto com mais atenção e garantir que as métricas estão realizando o calculo da forma adequada,
- Aparentemente os valores estão normalizados (0,1). Avaliar qual técnica foi utilizada (normalização?)

In [9]:
# --- Módulo D: avaliação de desempenho ---
metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
avaliacao.gerar_curva_perda(trainer.state.log_history)
avaliacao.salvar_resultados({**metricas, **eficiencia})

Curva de perda salva em: resultados/curva_perda.png
Resumo salvo em: resultados/resultado_final.csv
